In [9]:

#1
product_analysis = df.groupby("Product").agg(
    Orders=("OrderID", "count"),
    NetSales=("NetSales", "sum")
)

product_analysis["OrderPct"] = (
    product_analysis["Orders"] / len(df) * 100
)

product_analysis["SalesPct"] = (
    product_analysis["NetSales"] / product_analysis["NetSales"].sum() * 100
)

print(product_analysis.sort_values("OrderPct", ascending=False))

            Orders    NetSales   OrderPct   SalesPct
Product                                             
Chair           68  4587508.45  12.363636  11.202335
Tablet          67  5389068.90  12.181818  13.159682
Mouse           64  4670583.15  11.636364  11.405197
Desk            61  3965744.30  11.090909   9.684036
Monitor         60  3622098.60  10.909091   8.844880
Laptop          58  4241946.45  10.545455  10.358499
Mobile          58  4132147.50  10.545455  10.090379
Keyboard        57  4788821.75  10.363636  11.693926
Headphones      57  5553441.35  10.363636  13.561067


In [10]:
#2
high_sales_low_orders = product_analysis[
    (product_analysis["SalesPct"] > product_analysis["SalesPct"].mean()) &
    (product_analysis["OrderPct"] < product_analysis["OrderPct"].mean())
]

print(high_sales_low_orders)

            Orders    NetSales   OrderPct   SalesPct
Product                                             
Headphones      57  5553441.35  10.363636  13.561067
Keyboard        57  4788821.75  10.363636  11.693926


In [11]:

#3
delivered_product = delivered.groupby("Product")["NetSales"].sum().sort_values(
    ascending=False
)

print(delivered_product)

# Percentage contribution
delivered_product_pct = (
    delivered_product / delivered_product.sum() * 100
)

print("\nRevenue Percentage:")
print(delivered_product_pct)

Product
Headphones    4509278.95
Tablet        4242789.85
Chair         4103181.75
Keyboard      4039097.00
Mouse         3883433.30
Laptop        3552368.35
Desk          3391009.55
Monitor       3176003.50
Mobile        2938799.00
Name: NetSales, dtype: float64

Revenue Percentage:
Product
Headphones    13.326883
Tablet        12.539292
Chair         12.126689
Keyboard      11.937291
Mouse         11.477237
Laptop        10.498795
Desk          10.021910
Monitor        9.386473
Mobile         8.685431
Name: NetSales, dtype: float64


In [14]:
#4

pareto = delivered.groupby("Product")["NetSales"].sum().sort_values(
    ascending=False
)

pareto_pct = pareto / pareto.sum() * 100
pareto_cumulative = pareto_pct.cumsum()

pareto_table = pd.DataFrame({
    "Revenue": pareto,
    "Revenue %": pareto_pct,
    "Cumulative Revenue %": pareto_cumulative
})

print(pareto_table)

               Revenue  Revenue %  Cumulative Revenue %
Product                                                
Headphones  4509278.95  13.326883             13.326883
Tablet      4242789.85  12.539292             25.866175
Chair       4103181.75  12.126689             37.992863
Keyboard    4039097.00  11.937291             49.930154
Mouse       3883433.30  11.477237             61.407390
Laptop      3552368.35  10.498795             71.906186
Desk        3391009.55  10.021910             81.928096
Monitor     3176003.50   9.386473             91.314569
Mobile      2938799.00   8.685431            100.000000


In [15]:
#5
classification = delivered.groupby("Product").agg(
    Revenue=("NetSales", "sum"),
    Demand=("OrderID", "count")
)

avg_revenue = classification["Revenue"].mean()
avg_demand = classification["Demand"].mean()

def classify(row):
    if row["Revenue"] >= avg_revenue and row["Demand"] >= avg_demand:
        return "High Revenue / High Demand"
    
    elif row["Revenue"] >= avg_revenue and row["Demand"] < avg_demand:
        return "High Revenue / Low Demand"
    
    elif row["Revenue"] < avg_revenue and row["Demand"] >= avg_demand:
        return "Low Revenue / High Demand"
    
    else:
        return "Low Revenue / Low Demand"

classification["Group"] = classification.apply(classify, axis=1)

print(classification)

               Revenue  Demand                       Group
Product                                                   
Chair       4103181.75      60  High Revenue / High Demand
Desk        3391009.55      52   Low Revenue / High Demand
Headphones  4509278.95      48   High Revenue / Low Demand
Keyboard    4039097.00      50   High Revenue / Low Demand
Laptop      3552368.35      44    Low Revenue / Low Demand
Mobile      2938799.00      47    Low Revenue / Low Demand
Monitor     3176003.50      49    Low Revenue / Low Demand
Mouse       3883433.30      52  High Revenue / High Demand
Tablet      4242789.85      55  High Revenue / High Demand


In [11]:

#6
price_variation = df.groupby("Product")["UnitPrice"].agg(
    ["min", "max", "mean", "std"]
)

price_variation = price_variation.sort_values(
    "std", ascending=False
)

print(price_variation)

            Orders  AverageDiscount  AverageQuantity  AverageNetSales
Product                                                              
Mobile          58         0.091379         2.534483     71243.922414
Desk            61         0.090984         2.163934     65012.201639
Chair           68         0.087500         2.485294     67463.359559
Headphones      57         0.085965         2.754386     97428.795614
Monitor         60         0.082500         2.100000     60368.310000
Laptop          58         0.081897         2.431034     73137.007759
Keyboard        57         0.080702         2.526316     84014.416667
Tablet          67         0.076119         2.611940     80433.864179
Mouse           64         0.072656         2.484375     72977.861719


In [16]:
#7

price_check = df.groupby("Product")["UnitPrice"].agg(
    ["min", "max", "mean", "std"]
)

price_check["PriceRange"] = (
    price_check["max"] - price_check["min"]
)

print(price_check.sort_values("PriceRange", ascending=False))

             min    max          mean           std  PriceRange
Product                                                        
Desk        1007  64876  33380.721311  19637.424676       63869
Tablet      1184  64828  31883.149254  19140.926143       63644
Keyboard    1687  64836  35881.017544  17274.058676       63149
Monitor     1136  64219  31872.583333  19916.499244       63083
Headphones  2161  64910  37623.315789  15961.475275       62749
Mouse       1682  63945  31767.984375  19442.272384       62263
Chair       1971  63656  31763.352941  18110.043771       61685
Laptop      1670  63092  32241.431034  18149.443782       61422
Mobile      3751  63341  31929.293103  16694.382985       59590


In [61]:
#8
df["PriceBand"] = pd.qcut(
    df["UnitPrice"],
    q=3,
    labels=["Low Price", "Medium Price", "High Price"]
)
delivered = df[df["OrderStatus"] == "Delivered"].copy()
price_band_analysis = delivered.groupby(
    "PriceBand",
    observed=True
).agg(
    Orders=("OrderID", "count"),
    Revenue=("NetSales", "sum")
)

price_band_analysis["RevenuePct"] = (
    price_band_analysis["Revenue"] /
    price_band_analysis["Revenue"].sum() * 100
)

print(price_band_analysis)

              Orders      Revenue  RevenuePct
PriceBand                                    
Low Price        153   4184682.95   12.367560
Medium Price     152  11082900.70   32.754798
High Price       152  18568377.60   54.877642


In [62]:
#9


price_status = df.groupby(
    ["PriceBand", "OrderStatus"],
    observed=True
).size().unstack(fill_value=0)

price_status["Total"] = price_status.sum(axis=1)

price_status["CancellationRate"] = (
    price_status.get("Cancelled", 0) /
    price_status["Total"] * 100
)

price_status["ReturnRate"] = (
    price_status.get("Returned", 0) /
    price_status["Total"] * 100
)

print(price_status[
    ["CancellationRate", "ReturnRate"]
])

OrderStatus   CancellationRate  ReturnRate
PriceBand                                 
Low Price            13.114754    3.278689
Medium Price         10.928962    6.010929
High Price           10.869565    6.521739


In [26]:
#10
price_status["Cancel_Return_Rate"] = (
    (price_status["Cancelled"] + price_status["Returned"])
    / price_status["Total"] * 100
)

print(price_status[
    ["CancellationRate", "ReturnRate", "Cancel_Return_Rate"]
])

OrderStatus   CancellationRate  ReturnRate  Cancel_Return_Rate
PriceBand                                                     
Low Price            13.114754    3.278689           16.393443
Medium Price         10.928962    6.010929           16.939891
High Price           10.869565    6.521739           17.391304


In [29]:
#11
quantity_delivery = df.groupby("Quantity").agg(
    TotalOrders=("OrderID", "count"),
    DeliveredOrders=("OrderStatus", lambda x: (x == "Delivered").sum())
)

quantity_delivery["DeliveryRate"] = (
    quantity_delivery["DeliveredOrders"] /
    quantity_delivery["TotalOrders"] * 100
)

print(quantity_delivery)

          TotalOrders  DeliveredOrders  DeliveryRate
Quantity                                            
1                 142              118     83.098592
2                 143              118     82.517483
3                 138              116     84.057971
4                 127              105     82.677165


In [32]:
#12
quantity_sales = delivered.groupby("Quantity").agg(
    Orders=("OrderID", "count"),
    AverageNetSales=("NetSales", "mean")
)

print(quantity_sales.sort_values(
    "AverageNetSales",
    ascending=False
))

          Orders  AverageNetSales
Quantity                         
4            105    123491.487619
3            116     89369.220259
2            118     57161.690678
1            118     31842.762712


In [33]:
#13
df["PurchaseType"] = df["Quantity"].apply(
    lambda x: "Single Purchase" if x == 1 else "Bulk Purchase"
)

discount_analysis = df.groupby("PurchaseType").agg(
    Orders=("OrderID", "count"),
    AverageQuantity=("Quantity", "mean"),
    AverageDiscount=("DiscountPct", "mean"),
    AverageNetSales=("NetSales", "mean")
)

print(discount_analysis)

                 Orders  AverageQuantity  AverageDiscount  AverageNetSales
PurchaseType                                                              
Bulk Purchase       408         2.960784         0.083211     89469.702819
Single Purchase     142         1.000000         0.083099     31321.983803


In [34]:
#14
bulk_products = df[df["Quantity"] > 1]

bulk_analysis = bulk_products.groupby("Product").agg(
    BulkOrders=("OrderID", "count"),
    AverageQuantity=("Quantity", "mean"),
    TotalQuantity=("Quantity", "sum")
)

bulk_analysis = bulk_analysis.sort_values(
    "BulkOrders",
    ascending=False
)

print(bulk_analysis)

            BulkOrders  AverageQuantity  TotalQuantity
Product                                               
Chair               54         2.870370            155
Tablet              52         3.076923            160
Mouse               50         2.900000            145
Headphones          45         3.222222            145
Mobile              44         3.022727            133
Laptop              43         2.930233            126
Keyboard            42         3.071429            129
Desk                40         2.775000            111
Monitor             38         2.736842            104


In [35]:
#15
discount_demand = df.groupby("Product").agg(
    Orders=("OrderID", "count"),
    AverageDiscount=("DiscountPct", "mean"),
    AverageQuantity=("Quantity", "mean"),
    AverageNetSales=("NetSales", "mean")
)

print(discount_demand.sort_values(
    "AverageDiscount",
    ascending=False
))

            Orders  AverageDiscount  AverageQuantity  AverageNetSales
Product                                                              
Mobile          58         0.091379         2.534483     71243.922414
Desk            61         0.090984         2.163934     65012.201639
Chair           68         0.087500         2.485294     67463.359559
Headphones      57         0.085965         2.754386     97428.795614
Monitor         60         0.082500         2.100000     60368.310000
Laptop          58         0.081897         2.431034     73137.007759
Keyboard        57         0.080702         2.526316     84014.416667
Tablet          67         0.076119         2.611940     80433.864179
Mouse           64         0.072656         2.484375     72977.861719


In [36]:
#16  DEMO
import pandas as pd

df = pd.read_csv("sales_customer_analytics.csv")

df["DiscountPct"] = df["DiscountPct"].fillna(0)

df["NetSales"] = (
    df["Quantity"] *
    df["UnitPrice"] *
    (1 - df["DiscountPct"])
)

df["PriceBand"] = pd.qcut(
    df["UnitPrice"],
    q=3,
    labels=["Low Price", "Medium Price", "High Price"]
)

print(df.head())

   OrderID   OrderDate  CustomerID CustomerName       City  Product  \
0     1001  2024-01-01         252        Pooja      Delhi   Laptop   
1     1002  2024-01-01         293    Siddharth     Mumbai   Mobile   
2     1003  2024-01-01         215        Megha  Hyderabad   Mobile   
3     1004  2024-01-01         272       Ananya  Hyderabad  Monitor   
4     1005  2024-01-02         261        Rohit    Chennai   Tablet   

      Category  Quantity  UnitPrice  DiscountPct PaymentMode OrderStatus  \
0  Electronics         1       5852         0.10         UPI    Returned   
1  Electronics         1      27922         0.05        Cash   Delivered   
2  Electronics         3      42376         0.15  Debit Card   Cancelled   
3  Electronics         1      61492         0.20         UPI   Delivered   
4  Electronics         1      18306         0.15        Cash   Cancelled   

   Rating  NetSales     PriceBand  
0     5.0    5266.8     Low Price  
1     5.0   26525.9  Medium Price  
2     5.

In [37]:

#17
product = df.groupby("Product").agg(
    Orders=("OrderID", "count"),
    NetSales=("NetSales", "sum")
)

product["OrderPct"] = product["Orders"] / len(df) * 100
product["SalesPct"] = product["NetSales"] / product["NetSales"].sum() * 100

result = product[
    (product["OrderPct"] > product["OrderPct"].mean()) &
    (product["SalesPct"] < product["SalesPct"].mean())
]

print(result)

Empty DataFrame
Columns: [Orders, NetSales, OrderPct, SalesPct]
Index: []


In [38]:
#18
result = product[
    (product["SalesPct"] > product["SalesPct"].mean()) &
    (product["OrderPct"] < product["OrderPct"].mean())
]

print(result)

            Orders    NetSales   OrderPct   SalesPct
Product                                             
Headphones      57  5553441.35  10.363636  13.561067
Keyboard        57  4788821.75  10.363636  11.693926


In [39]:
#19
delivered = df[df["OrderStatus"] == "Delivered"]

revenue = delivered.groupby("Product")["NetSales"].sum()

revenue_pct = revenue / revenue.sum() * 100

result = pd.DataFrame({
    "Delivered Revenue": revenue,
    "Revenue %": revenue_pct
}).sort_values("Delivered Revenue", ascending=False)

print(result)


            Delivered Revenue  Revenue %
Product                                 
Headphones         4509278.95  13.326883
Tablet             4242789.85  12.539292
Chair              4103181.75  12.126689
Keyboard           4039097.00  11.937291
Mouse              3883433.30  11.477237
Laptop             3552368.35  10.498795
Desk               3391009.55  10.021910
Monitor            3176003.50   9.386473
Mobile             2938799.00   8.685431


In [40]:
#20
pareto = delivered.groupby("Product")["NetSales"].sum()
pareto = pareto.sort_values(ascending=False)

pareto["Cumulative %"] = (
    pareto.cumsum() / pareto.sum() * 100
)

print(pareto)

Product
Headphones                                             4509278.95
Tablet                                                 4242789.85
Chair                                                  4103181.75
Keyboard                                                4039097.0
Mouse                                                   3883433.3
Laptop                                                 3552368.35
Desk                                                   3391009.55
Monitor                                                 3176003.5
Mobile                                                  2938799.0
Cumulative %    Product
Headphones     13.326883
Tablet       ...
Name: NetSales, dtype: object


In [41]:
#21
classification = delivered.groupby("Product").agg(
    Revenue=("NetSales", "sum"),
    Demand=("OrderID", "count")
)

avg_revenue = classification["Revenue"].mean()
avg_demand = classification["Demand"].mean()

def classify(row):
    if row["Revenue"] >= avg_revenue and row["Demand"] >= avg_demand:
        return "High Revenue / High Demand"
    elif row["Revenue"] >= avg_revenue and row["Demand"] < avg_demand:
        return "High Revenue / Low Demand"
    elif row["Revenue"] < avg_revenue and row["Demand"] >= avg_demand:
        return "Low Revenue / High Demand"
    else:
        return "Low Revenue / Low Demand"

classification["Group"] = classification.apply(classify, axis=1)

print(classification)

               Revenue  Demand                       Group
Product                                                   
Chair       4103181.75      60  High Revenue / High Demand
Desk        3391009.55      52   Low Revenue / High Demand
Headphones  4509278.95      48   High Revenue / Low Demand
Keyboard    4039097.00      50   High Revenue / Low Demand
Laptop      3552368.35      44    Low Revenue / Low Demand
Mobile      2938799.00      47    Low Revenue / Low Demand
Monitor     3176003.50      49    Low Revenue / Low Demand
Mouse       3883433.30      52  High Revenue / High Demand
Tablet      4242789.85      55  High Revenue / High Demand


In [42]:

#22
price_variation = df.groupby("Product")["UnitPrice"].agg(
    ["min", "max", "mean", "std"]
)

price_variation["PriceRange"] = (
    price_variation["max"] - price_variation["min"]
)

print(price_variation.sort_values(
    "std",
    ascending=False
))


             min    max          mean           std  PriceRange
Product                                                        
Monitor     1136  64219  31872.583333  19916.499244       63083
Desk        1007  64876  33380.721311  19637.424676       63869
Mouse       1682  63945  31767.984375  19442.272384       62263
Tablet      1184  64828  31883.149254  19140.926143       63644
Laptop      1670  63092  32241.431034  18149.443782       61422
Chair       1971  63656  31763.352941  18110.043771       61685
Keyboard    1687  64836  35881.017544  17274.058676       63149
Mobile      3751  63341  31929.293103  16694.382985       59590
Headphones  2161  64910  37623.315789  15961.475275       62749


In [44]:
#23
price_check = df.groupby("Product")["UnitPrice"].agg(
    ["min", "max", "mean", "std"]
)

price_check["PriceDifference"] = (
    price_check["max"] - price_check["min"]
)

print(price_check.sort_values(
    "PriceDifference",
    ascending=False
))

             min    max          mean           std  PriceDifference
Product                                                             
Desk        1007  64876  33380.721311  19637.424676            63869
Tablet      1184  64828  31883.149254  19140.926143            63644
Keyboard    1687  64836  35881.017544  17274.058676            63149
Monitor     1136  64219  31872.583333  19916.499244            63083
Headphones  2161  64910  37623.315789  15961.475275            62749
Mouse       1682  63945  31767.984375  19442.272384            62263
Chair       1971  63656  31763.352941  18110.043771            61685
Laptop      1670  63092  32241.431034  18149.443782            61422
Mobile      3751  63341  31929.293103  16694.382985            59590


In [45]:
#24
price_band = delivered.groupby(
    "PriceBand",
    observed=True
).agg(
    Orders=("OrderID", "count"),
    Revenue=("NetSales", "sum")
)

price_band["RevenuePct"] = (
    price_band["Revenue"] /
    price_band["Revenue"].sum() * 100
)

print(price_band)

              Orders      Revenue  RevenuePct
PriceBand                                    
Low Price        153   4184682.95   12.367560
Medium Price     152  11082900.70   32.754798
High Price       152  18568377.60   54.877642


In [46]:
#25
status = df.groupby(
    ["PriceBand", "OrderStatus"],
    observed=True
).size().unstack(fill_value=0)

status["Total"] = status.sum(axis=1)

status["CancellationRate"] = (
    status.get("Cancelled", 0) /
    status["Total"] * 100
)

status["ReturnRate"] = (
    status.get("Returned", 0) /
    status["Total"] * 100
)

print(status[
    ["CancellationRate", "ReturnRate"]
])

OrderStatus   CancellationRate  ReturnRate
PriceBand                                 
Low Price            13.114754    3.278689
Medium Price         10.928962    6.010929
High Price           10.869565    6.521739


In [53]:
#26
status["Cancel_Return_Rate"] = (
    (status.get("Cancelled", 0) +
     status.get("Returned", 0))
    / status["Total"] * 100
)

print(status[
    ["CancellationRate", "ReturnRate", "Cancel_Return_Rate"]
])

OrderStatus   CancellationRate  ReturnRate  Cancel_Return_Rate
PriceBand                                                     
Low Price            13.114754    3.278689           16.393443
Medium Price         10.928962    6.010929           16.939891
High Price           10.869565    6.521739           17.391304


In [54]:
#27
quantity_delivery = df.groupby("Quantity").agg(
    TotalOrders=("OrderID", "count"),
    DeliveredOrders=(
        "OrderStatus",
        lambda x: (x == "Delivered").sum()
    )
)

quantity_delivery["DeliveryRate"] = (
    quantity_delivery["DeliveredOrders"] /
    quantity_delivery["TotalOrders"] * 100
)

print(quantity_delivery)

          TotalOrders  DeliveredOrders  DeliveryRate
Quantity                                            
1                 142              118     83.098592
2                 143              118     82.517483
3                 138              116     84.057971
4                 127              105     82.677165


In [55]:
#28
quantity_sales = delivered.groupby("Quantity").agg(
    Orders=("OrderID", "count"),
    AverageNetSales=("NetSales", "mean")
)

print(quantity_sales.sort_values(
    "AverageNetSales",
    ascending=False
))

          Orders  AverageNetSales
Quantity                         
4            105    123491.487619
3            116     89369.220259
2            118     57161.690678
1            118     31842.762712


In [56]:
#29
df["PurchaseType"] = df["Quantity"].apply(
    lambda x: "Single Quantity" if x == 1
    else "Bulk Purchase"
)

discount = df.groupby("PurchaseType").agg(
    Orders=("OrderID", "count"),
    AverageQuantity=("Quantity", "mean"),
    AverageDiscount=("DiscountPct", "mean"),
    AverageNetSales=("NetSales", "mean")
)

print(discount)

                 Orders  AverageQuantity  AverageDiscount  AverageNetSales
PurchaseType                                                              
Bulk Purchase       408         2.960784         0.083211     89469.702819
Single Quantity     142         1.000000         0.083099     31321.983803


In [57]:
#30
bulk = df[df["Quantity"] > 1]

bulk_products = bulk.groupby("Product").agg(
    BulkOrders=("OrderID", "count"),
    TotalQuantity=("Quantity", "sum"),
    AverageQuantity=("Quantity", "mean")
)

print(bulk_products.sort_values(
    "BulkOrders",
    ascending=False
))

            BulkOrders  TotalQuantity  AverageQuantity
Product                                               
Chair               54            155         2.870370
Tablet              52            160         3.076923
Mouse               50            145         2.900000
Headphones          45            145         3.222222
Mobile              44            133         3.022727
Laptop              43            126         2.930233
Keyboard            42            129         3.071429
Desk                40            111         2.775000
Monitor             38            104         2.736842


In [58]:
#31
discount_demand = df.groupby("Product").agg(
    Orders=("OrderID", "count"),
    AverageDiscount=("DiscountPct", "mean"),
    AverageQuantity=("Quantity", "mean"),
    AverageNetSales=("NetSales", "mean")
)

print(discount_demand.sort_values(
    "AverageDiscount",
    ascending=False
))

            Orders  AverageDiscount  AverageQuantity  AverageNetSales
Product                                                              
Mobile          58         0.091379         2.534483     71243.922414
Desk            61         0.090984         2.163934     65012.201639
Chair           68         0.087500         2.485294     67463.359559
Headphones      57         0.085965         2.754386     97428.795614
Monitor         60         0.082500         2.100000     60368.310000
Laptop          58         0.081897         2.431034     73137.007759
Keyboard        57         0.080702         2.526316     84014.416667
Tablet          67         0.076119         2.611940     80433.864179
Mouse           64         0.072656         2.484375     72977.861719


In [64]:
#32
df["PriceBand"] = pd.qcut(
    df["UnitPrice"],
    q=3,
    labels=["Low", "Medium", "Premium"]
)


premium_city = df[df["PriceBand"] == "Premium"].groupby("City").agg(
    PremiumOrders=("OrderID", "count"),
    TotalQuantity=("Quantity", "sum")
)

print(premium_city.sort_values(
    "PremiumOrders",
    ascending=False
))

           PremiumOrders  TotalQuantity
City                                   
Chennai               34             93
Bangalore             33             83
Delhi                 33             86
Hyderabad             30             68
Mumbai                29             73
Pune                  25             58


In [72]:
#33
discount_city = df.groupby("City").agg(
    TotalOrders=("OrderID", "count"),
    DiscountedOrders=(
        "DiscountPct",
        lambda x: (x > 0).sum()
    )
)

discount_city["DiscountedOrderPct"] = (
    discount_city["DiscountedOrders"] /
    discount_city["TotalOrders"] * 100
)

print(discount_city.sort_values(
    "DiscountedOrderPct",
    ascending=False
))

           TotalOrders  DiscountedOrders  DiscountedOrderPct
City                                                        
Bangalore           93                73           78.494624
Pune                97                66           68.041237
Delhi               95                64           67.368421
Hyderabad           81                53           65.432099
Mumbai              82                53           64.634146
Chennai            102                62           60.784314


In [73]:
#34
city_payment = pd.crosstab(
    df["City"],
    df["PaymentMode"],
    normalize="index"
) * 100

print(city_payment.round(2))

PaymentMode   Cash  Credit Card  Debit Card    UPI
City                                              
Bangalore    21.51        24.73       21.51  32.26
Chennai      29.41        25.49       22.55  22.55
Delhi        27.37        23.16       23.16  26.32
Hyderabad    27.16        27.16       19.75  25.93
Mumbai       24.39        25.61       18.29  31.71
Pune         25.77        16.49       25.77  31.96


In [74]:
#35
city_payment_status = df.groupby(
    ["City", "PaymentMode"]
).agg(
    TotalOrders=("OrderID", "count"),
    Cancelled=("OrderStatus", lambda x: (x == "Cancelled").sum()),
    Returned=("OrderStatus", lambda x: (x == "Returned").sum())
)

city_payment_status["CancellationRate"] = (
    city_payment_status["Cancelled"] /
    city_payment_status["TotalOrders"] * 100
)

city_payment_status["ReturnRate"] = (
    city_payment_status["Returned"] /
    city_payment_status["TotalOrders"] * 100
)

city_payment_status["Cancel_Return_Rate"] = (
    (city_payment_status["Cancelled"] +
     city_payment_status["Returned"]) /
    city_payment_status["TotalOrders"] * 100
)

print(
    city_payment_status.sort_values(
        "Cancel_Return_Rate",
        ascending=False
    )
)

                       TotalOrders  Cancelled  Returned  CancellationRate  \
City      PaymentMode                                                       
Hyderabad Debit Card            16          4         1         25.000000   
Mumbai    Cash                  20          4         2         20.000000   
Hyderabad Credit Card           22          5         1         22.727273   
Pune      Credit Card           16          1         3          6.250000   
Chennai   Cash                  30          3         4         10.000000   
Delhi     Cash                  26          5         1         19.230769   
          Debit Card            22          4         1         18.181818   
Bangalore Cash                  20          2         2         10.000000   
Mumbai    Debit Card            15          3         0         20.000000   
Bangalore Debit Card            20          2         2         10.000000   
Chennai   Credit Card           26          4         1         15.384615   

In [75]:
#36
high_value_limit = df["NetSales"].quantile(0.75)

high_value = df[df["NetSales"] >= high_value_limit]

payment_high_value = high_value.groupby("PaymentMode").agg(
    HighValueOrders=("OrderID", "count"),
    AverageOrderValue=("NetSales", "mean")
)

print(payment_high_value.sort_values(
    "HighValueOrders",
    ascending=False
))

             HighValueOrders  AverageOrderValue
PaymentMode                                    
Cash                      41      160527.119512
Credit Card               35      168467.075714
UPI                       33      159195.443939
Debit Card                29      155149.798276


In [76]:
#37

high_value_limit = df["NetSales"].quantile(0.75)

high_value = df[df["NetSales"] >= high_value_limit]

payment_high_value = high_value.groupby("PaymentMode").agg(
    HighValueOrders=("OrderID", "count"),
    AverageOrderValue=("NetSales", "mean")
)

print(payment_high_value.sort_values(
    "HighValueOrders",
    ascending=False
))

             HighValueOrders  AverageOrderValue
PaymentMode                                    
Cash                      41      160527.119512
Credit Card               35      168467.075714
UPI                       33      159195.443939
Debit Card                29      155149.798276


In [77]:
#38
df["PaymentType"] = df["PaymentMode"].apply(
    lambda x: "Cash" if x == "Cash" else "Digital"
)

payment_type = df.groupby("PaymentType").agg(
    Orders=("OrderID", "count"),
    AverageOrderValue=("NetSales", "mean")
)

print(payment_type)

             Orders  AverageOrderValue
PaymentType                           
Cash            143       77606.955944
Digital         407       73350.284398


In [78]:
#39
monthly_payment = df.groupby(
    [df["OrderDate"].dt.to_period("M"), "PaymentMode"]
).size().unstack(fill_value=0)

print(monthly_payment)


PaymentMode  Cash  Credit Card  Debit Card  UPI
OrderDate                                      
2024-01        27           18          19   31
2024-02        22           21          17   27
2024-03        25           25          20   25
2024-04        18           21          24   28
2024-05        26           21          22   25
2024-06        25           24          19   20


In [80]:
#40
payment_analysis = df.groupby("PaymentMode").agg(
    TotalOrders=("OrderID", "count"),
    AverageOrderValue=("NetSales", "mean"),
    DeliveredOrders=(
        "OrderStatus",
        lambda x: (x == "Delivered").sum()
    )
)

payment_analysis["DeliveryRate"] = (
    payment_analysis["DeliveredOrders"] /
    payment_analysis["TotalOrders"] * 100
)

print(payment_analysis)

             TotalOrders  AverageOrderValue  DeliveredOrders  DeliveryRate
PaymentMode                                                               
Cash                 143       77606.955944              116     81.118881
Credit Card          130       79821.715385              107     82.307692
Debit Card           121       71944.041322               97     80.165289
UPI                  156       69048.165064              137     87.820513


In [81]:
#41
customer_orders = df.groupby("CustomerID")["OrderID"].count()

once = (customer_orders == 1).sum()
twice = (customer_orders == 2).sum()
three_or_more = (customer_orders >= 3).sum()

print("Purchased once:", once)
print("Purchased twice:", twice)
print("Purchased 3 or more times:", three_or_more)

Purchased once: 3
Purchased twice: 5
Purchased 3 or more times: 92


In [83]:
#42
# Customers with 3 or more orders
customer_order_count = df.groupby("CustomerID")["OrderID"].count()

repeat_customers = customer_order_count[
    customer_order_count >= 3
].index

# Delivered orders only
delivered = df[df["OrderStatus"] == "Delivered"]

# Total delivered revenue
total_delivered_revenue = delivered["NetSales"].sum()

# Delivered revenue from 3+ order customers
repeat_delivered_revenue = delivered[
    delivered["CustomerID"].isin(repeat_customers)
]["NetSales"].sum()

percentage = (
    repeat_delivered_revenue /
    total_delivered_revenue
) * 100

print("Percentage of delivered revenue:", percentage)

Percentage of delivered revenue: 97.57902636798889
